# import

In [ ]:
from utils.recbole_train_test import *
from utils.plot_utils import *
from utils.model_utils import get_trainer

# Pre train Goodreads

In [ ]:
save_path, base_filename, specs_str = ('processed_datasets/natural_data/goodreads/inter_dedup_coldstart_3stars_4x714k/',
                                        'more_2interQ_df',
                                        'PT')

base_dataset_name = base_filename+'_'+specs_str

# MODEL_VERSIONS = ['_pt1', '_pt2', '_pt3', '_pt4']
freq=6 # month
duration = 2*12//freq # 2 years split in xM buckets
n_parts = duration*2+1
d_keys = ['_pt'+str(i) for i in range(1, n_parts)]
MODEL_VERSIONS = d_keys[:duration]


K = [1, 10, 20]
VM_K = 2 # valid metric k, also used in heatmap matrix
VALID_METRIC = 'Recall@'+str(K[VM_K])
SEED = 2020
USE_GPU = False
SHOW_PROGRESS = False

# these are the default values
# TRAIN_NEG_SAMPLE_ARGS = {'distribution': 'uniform', 
#                          'sample_num': 1, 
#                          'alpha': 1.0, 
#                          'dynamic': False, 
#                          'candidate_num': 0}



SHUFFLE = False  # shuffle (bool): Whether or not to shuffle the training data before each epoch. Defaults to True.
EVAL_ARGS = {'split': {'LS': 'test_only'}, # leave-one-out sample type ['valid_and_test', 'valid_only', 'test_only']
                    'group_by': 'user',
                    'order': 'TO', # order (str): decides how we sort the data in .inter. random ordering or time ordering
                    'mode': 'uni100'}

FILENAME_VERSION = '_ET_LS.t_UD_SF_TO_UM.100'

## BPR

In [ ]:
model_name = 'BPR'


for part in MODEL_VERSIONS:
    print('\n\n'+part)
    dataset_name=base_dataset_name+part
    parameter_dict = {
        'dataset': dataset_name+'.inter',
        'use_gpu':USE_GPU,
        ##########################################################################################
        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': save_path,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':save_path+dataset_name,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ##########################################################################################
        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ##########################################################################################
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        

        ##########################################################################################
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }

    recbole_train_bpr(model_name, dataset_name, parameter_dict, FILENAME_VERSION)


## Experiment Goodreads - ET RD LS.t UD SF TO UM.100
executed train, random drift, leave one out train sampled, uniform neg sample training distribution, shuffle false, time ordering, uniform mode


In [ ]:
save_path, base_filename, specs_str = ('processed_datasets/natural_data/goodreads/inter_dedup_coldstart_3stars_4x714k/',
                                        'more_2interQ_df', 
                                        'NPT_NT_RD.50')
base_dataset_name = base_filename+'_'+specs_str

model_name = 'BPR'



for model_part in MODEL_VERSIONS:
    print('\n\n'+model_part)
    dataset_name=base_dataset_name+model_part
    parameter_dict = {
        'dataset': dataset_name+'.inter',
        'use_gpu':USE_GPU,
        ##########################################################################################
        ## Environment settings https://recbole.io/docs/user_guide/config/environment_settings.html
        'seed':SEED,
        'state':'ERROR',
        'data_path': save_path,
        # save_dataset (bool): Whether or not to save filtered dataset. If True, save filtered dataset, otherwise it will not be saved. Defaults to False
        'checkpoint_dir':save_path+dataset_name,
        # 'show_progress': SHOW_PROGRESS,
        'shuffle': SHUFFLE,

        ##########################################################################################
        ## Data settings https://recbole.io/docs/user_guide/config/data_settings.html
        'load_col': {'inter': ['user_id', 'item_id', 'timestamp']},
        # 'user_inter_num_interval':'[1,inf)',
        
        ##########################################################################################
        ## Training settings https://recbole.io/docs/user_guide/config/training_settings.html
        

        ##########################################################################################
        ## Evaluation settings https://recbole.io/docs/user_guide/config/evaluation_settings.html
        'eval_args': EVAL_ARGS,
        # 'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'], # default
        'topk':K,
        'valid_metric':VALID_METRIC          
    }

    model_config, model_trainer = recbole_train_bpr(model_name, dataset_name, parameter_dict, FILENAME_VERSION)

    testset_list = get_test_full_data_sections_with_names(model_part, base_dataset_name, MODEL_VERSIONS)
    for testset_name in testset_list:

        _,_,\
            _,_,_,\
                test_data = setup_config_and_dataset(model_name,
                                                     testset_name,
                                                     parameter_dict)

        test_result = evaluate(model_trainer, test_data)
        save_evaluation_results(test_result, 
                                parameter_dict['checkpoint_dir'], 
                                get_evaluation_results_filename(model_config['model'], 
                                                                dataset_name, 
                                                                testset_name, 
                                                                FILENAME_VERSION))


### recall heatmap

In [ ]:
save_path, base_filename, specs_str = ('processed_datasets/natural_data/goodreads/inter_dedup_coldstart_3stars_4x714k/',
                                        'more_2interQ_df', 
                                        'NPT_NT_RD.50')
base_dataset_name = base_filename+'_'+specs_str

model_name = 'BPR'

results_matrix = get_results_matrix(model_name=model_name,
                                    models_versions=MODEL_VERSIONS,
                                    base_dataset_name=base_dataset_name,
                                    save_path=save_path,
                                    metric='recall@'+str(K[VM_K]),
                                    filename_version=FILENAME_VERSION,
                                    part_shift_incl=False,
                                    test_full_data_sec=True)

recall_heatmap(results_matrix, 
               round_point=4, 
               title='BPR - Goodreads - 50\% Random Drift', 
               filepath='images/goodreads/'+base_dataset_name+FILENAME_VERSION+'_'+model_name)